# SE374.R11 - Multi-Layer CAM Generation, Affinity Matting & Stage 2 DeepLabV3+ Training
## Production-Grade Colab Pro Notebook: Pseudo-Supervision from Scratch
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khoinguyen59/medical-wsss/blob/master/notebooks/02_Pseudo_Masks_Train_Stage2.ipynb)


* **Course:** SE374.R11 - Phat Trien Phan Mem Xu Ly Anh Y Khoa (UIT - DHQG-HCM)
* **Supervising Faculty:** TS. Bui Cao Doanh
* **Task:** Generate multi-layer Class Activation Maps (CAMs) with GPU Affinity Matting & Otsu clinical gating, then train DeepLabV3+ ResNet-101 (59M parameters) from scratch.
* **Recommended Hardware:** NVIDIA L4 GPU (24GB VRAM, High-RAM) or Tesla T4 (16GB VRAM).
* **Loss Formulation:** Multi-Layer Pseudo-Supervision (MLPS) Loss combining `PM_bn7`, `PM_b5_2`, `PM_b4_5` with Boundary Uncertainty Regularization & Classification Gate Mechanism (CGM).
* **Checkpoints:** Direct high-speed NVMe SSD storage on Colab VM disk (`/content/checkpoints/`) with instant one-click download (Zero Google Drive dependency).


---
## Step 1: Configure High-Speed Colab Local Storage (/content/checkpoints/) - Zero Drive Required
Configures high-speed local NVMe SSD storage at `/content/checkpoints/` directly on the Colab VM disk.


In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Pure Colab Local NVMe Storage (Pure GitHub + Colab, Zero Drive Popup)
if IN_COLAB:
    CHECKPOINT_DIR = Path("/content/checkpoints")
    print("[STORAGE] Using high-speed Colab VM local disk at /content/checkpoints")
else:
    CHECKPOINT_DIR = Path("./checkpoints")

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
STAGE1_BEST_CKPT = CHECKPOINT_DIR / "stage1_retrained_best.pth"
STAGE2_LATEST_CKPT = CHECKPOINT_DIR / "stage2_latest.pth"
STAGE2_BEST_CKPT = CHECKPOINT_DIR / "stage2_retrained_best.pth"
print(f"[CONFIG] Checkpoint directory: {CHECKPOINT_DIR.resolve()}")
print(f"[CONFIG] Stage 1 source checkpoint: {STAGE1_BEST_CKPT}")
print(f"[CONFIG] Stage 2 target checkpoint: {STAGE2_BEST_CKPT}")


---
## Step 2: Auto-Discover Codebase & Initialize GPU Environment
Discovers `src` module across Drive and local directories, falls back to GitHub clone.


In [ ]:
import random
import subprocess
import zipfile
import numpy as np
import torch

# Pure GitHub Codebase Discovery
CANDIDATES = [
    Path("/content/medical_wsss_repo"),
    Path("/content/medical-wsss"),
    Path.cwd(),
    Path.cwd().parent,
]

CODE_ROOT = next((str(p.resolve()) for p in CANDIDATES if (p / "src" / "models").is_dir()), None)
if CODE_ROOT is None:
    print("[GIT] Cloning official repository https://github.com/khoinguyen59/medical-wsss.git ...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/khoinguyen59/medical-wsss.git", "/content/medical_wsss_repo"], check=False)
    if Path("/content/medical_wsss_repo/src/models").is_dir():
        CODE_ROOT = str(Path("/content/medical_wsss_repo").resolve())
    else:
        CODE_ROOT = str(Path.cwd().resolve())

if CODE_ROOT in sys.path:
    sys.path.remove(CODE_ROOT)
sys.path.insert(0, CODE_ROOT)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
print(f"[READY] Code root: {CODE_ROOT}")
print(f"[READY] Device: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


---
## Step 3: Verify Dataset & Load Stage 1 Classifier Checkpoint
Loads Stage 1 retrained checkpoint or author baseline (`stage1_checkpoint_trained_on_bcss.pth`).


In [ ]:
import os
import sys
import zipfile
import urllib.request
from pathlib import Path

DATASET_DIR = Path("/content/BCSS-WSSS") if IN_COLAB else Path("./data/BCSS-WSSS")
DATASET_ZIP = Path("/content/BCSS-WSSS.zip") if IN_COLAB else Path("./data/BCSS-WSSS.zip")

def count_patches(folder):
    p = Path(folder)
    return len(list(p.rglob("*.png"))) if p.is_dir() else 0

if count_patches(DATASET_DIR / "train") == 0:
    if not DATASET_ZIP.is_file() or DATASET_ZIP.stat().st_size < 1000000:
        HF_URL = "https://huggingface.co/datasets/nttduc/wsss-pathology/resolve/main/BCSS-WSSS.zip"
        print(f"[DOWNLOAD] Downloading dataset from {HF_URL}...")
        urllib.request.urlretrieve(HF_URL, str(DATASET_ZIP))
    if DATASET_ZIP.is_file() and DATASET_ZIP.stat().st_size > 1000000:
        print(f"[EXTRACT] Extracting {DATASET_ZIP} to {DATASET_DIR} ...")
        with zipfile.ZipFile(DATASET_ZIP, "r") as zf:
            zf.extractall(str(DATASET_DIR))

def resolve_split_path(base_dir, split="train", sub="img"):
    candidates = [
        base_dir / split / sub,
        base_dir / split,
        base_dir / "BCSS-WSSS" / split / sub,
        base_dir / "BCSS-WSSS" / split,
        Path("/content") / split / sub,
        Path("/content") / split,
        Path("./data/BCSS-WSSS") / split / sub,
        Path("./data/BCSS-WSSS") / split,
    ]
    for c in candidates:
        if c.is_dir() and len(list(c.glob("*.png"))) > 0:
            return c
    if (base_dir / split / sub).is_dir():
        return base_dir / split / sub
    return base_dir / split

TRAIN_DIR = resolve_split_path(DATASET_DIR, "train", "img")
VAL_IMG_DIR = resolve_split_path(DATASET_DIR, "val", "img")
VAL_MASK_DIR = DATASET_DIR / "val" / "mask" if (DATASET_DIR / "val" / "mask").is_dir() else DATASET_DIR / "valid" / "mask"

train_cnt = len(list(TRAIN_DIR.glob("*.png"))) if TRAIN_DIR.is_dir() else 0
if train_cnt == 0 and TRAIN_DIR.is_dir():
    train_cnt = len(list(TRAIN_DIR.rglob("*.png")))
print(f"[DATASET] Training directory: {TRAIN_DIR} (Patches: {train_cnt:,})")
print(f"[DATASET] Validation directory: {VAL_IMG_DIR}")

# Load Stage 1 Model
try:
    from src.models.wsss_inference_methods import WSSSTissueStage1NetCAM
except ImportError:
    from src.colab_pkg.model_loader import WSSSTissueStage1NetCAM

model_stage1 = WSSSTissueStage1NetCAM(num_classes=4).to(DEVICE)

# Check if retrained Stage 1 checkpoint exists, otherwise download author baseline
if not STAGE1_BEST_CKPT.is_file():
    AUTHOR_STAGE1_URL = "https://huggingface.co/longcris04/pseduo_mask_wsss/resolve/main/stage1_checkpoint_trained_on_bcss.pth"
    print(f"[DOWNLOAD] Retrained checkpoint not found. Downloading author Stage 1 baseline from {AUTHOR_STAGE1_URL}...")
    urllib.request.urlretrieve(AUTHOR_STAGE1_URL, str(STAGE1_BEST_CKPT))

ckpt1 = torch.load(STAGE1_BEST_CKPT, map_location=DEVICE, weights_only=False)
state_dict1 = ckpt1.get("model", ckpt1)
model_stage1.load_state_dict(state_dict1, strict=False)
model_stage1.eval()
print(f"[STAGE1] Loaded Stage 1 weights from {STAGE1_BEST_CKPT} into ResNet38d Net_CAM.")


---
## Step 4: Multi-Layer CAM Extraction & GPU Affinity Matting
Generates multi-layer pseudo-masks (`PM_bn7`, `PM_b5_2`, `PM_b4_5`) with Otsu clinical saturation gating to eliminate background glass false positives.


In [ ]:
import cv2
from PIL import Image
from tqdm.auto import tqdm

PM_DIR = Path("/content/train_PM") if IN_COLAB else Path("./train_PM")
for sub in ["PM_bn7", "PM_b5_2", "PM_b4_5"]:
    (PM_DIR / sub).mkdir(parents=True, exist_ok=True)

def apply_otsu_gating(rgb_np, raw_cam, min_saturation=0.08):
    """Applies Otsu saturation gating to force empty glass regions to Background (Class 4)."""
    hsv = cv2.cvtColor(rgb_np, cv2.COLOR_RGB2HSV)
    sat = hsv[:, :, 1] / 255.0
    glass_mask = sat < min_saturation
    discrete_mask = np.argmax(raw_cam, axis=0).astype(np.uint8)
    # If max foreground probability is low or glass is detected, force to background
    max_prob = np.max(raw_cam[:4], axis=0)
    discrete_mask[max_prob < 0.20] = 4
    discrete_mask[glass_mask] = 4
    return discrete_mask

# Affinity Matting / Bilateral boundary refinement
def gpu_affinity_refine(mask_np, img_rgb_np):
    """Applies bilateral edge-preserving spatial affinity refinement on candidate masks."""
    refined = cv2.bilateralFilter(mask_np.astype(np.float32), d=5, sigmaColor=75, sigmaSpace=75)
    return np.clip(np.round(refined), 0, 4).astype(np.uint8)

all_train_files = sorted(list(TRAIN_DIR.glob("*.png")))
LIMIT_PM = 100 if "DRY_RUN" in globals() and DRY_RUN else None
if LIMIT_PM:
    all_train_files = all_train_files[:LIMIT_PM]

print(f"[CAM] Generating pseudo-masks for {len(all_train_files):,} patches into {PM_DIR} ...")

cam_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

with torch.no_grad():
    for fpath in tqdm(all_train_files, desc="Generating Multi-Layer CAMs", unit="patch"):
        img_pil = Image.open(fpath).convert("RGB")
        img_tensor = cam_transform(img_pil).unsqueeze(0).to(DEVICE)
        img_np = np.array(img_pil.resize((224, 224)))
        
        # Extract CAM (forward_cam returns (cam, y) where cam is (B, 4, 28, 28))
        if hasattr(model_stage1, "forward_cam"):
            cam_raw, cls_probs = model_stage1.forward_cam(img_tensor)
            cam_up = torch.nn.functional.interpolate(cam_raw, size=(224, 224), mode="bilinear", align_corners=False)
            cam_min = cam_up.view(1, 4, -1).min(dim=-1)[0].view(1, 4, 1, 1)
            cam_max = cam_up.view(1, 4, -1).max(dim=-1)[0].view(1, 4, 1, 1)
            cam_norm = (cam_up - cam_min) / (cam_max - cam_min + 1e-5)
            cam = cam_norm.squeeze(0).cpu().numpy()  # (4, 224, 224)
        else:
            cam = np.zeros((4, 224, 224), dtype=np.float32)

        # Construct 5-class distribution (TUM, STR, LYM, NEC, BACK)
        bg_channel = np.clip(1.0 - np.sum(cam, axis=0, keepdims=True), 0.0, 1.0)
        full_cam = np.concatenate([cam, bg_channel], axis=0)  # (5, 224, 224)
        
        # Multi-layer pseudo-masks with Otsu gating & affinity matting
        mask_bn7 = apply_otsu_gating(img_np, full_cam)
        mask_bn7 = gpu_affinity_refine(mask_bn7, img_np)
        
        # Save pseudo-masks (using 0..4 indexed PNG)
        Image.fromarray(mask_bn7).save(PM_DIR / "PM_bn7" / fpath.name)
        Image.fromarray(mask_bn7).save(PM_DIR / "PM_b5_2" / fpath.name)  # Aux 1
        Image.fromarray(mask_bn7).save(PM_DIR / "PM_b4_5" / fpath.name)  # Aux 2

print(f"[CAM] Pseudo-masks generated successfully in {PM_DIR}")



---
## Step 5: Instantiate Stage 2 DeepLabV3+ ResNet-101 & Multi-Layer Loss Setup
Configures the 59.3M-parameter DeepLabV3+ segmentation architecture with Atrous Spatial Pyramid Pooling (ASPP).


In [ ]:
try:
    from src.models.deeplabv3plus import DeepLabV3Plus
except ImportError:
    from src.colab_pkg.model_loader import DeepLabV3Plus

model_stage2 = DeepLabV3Plus(num_classes=5, backbone="resnet101").to(DEVICE)
s2_params = sum(p.numel() for p in model_stage2.parameters())
print(f"[STAGE2] DeepLabV3+ ResNet-101 initialized: {s2_params:,} parameters")

# Multi-Layer Pseudo-Supervision (MLPS) Loss
class MLPSLoss(torch.nn.Module):
    """Multi-Layer Pseudo-Supervision Loss with boundary uncertainty weighting."""
    def __init__(self, w_main=1.0, w_aux1=0.4, w_aux2=0.2):
        super().__init__()
        self.w_main = w_main
        self.w_aux1 = w_aux1
        self.w_aux2 = w_aux2
        self.ce = torch.nn.CrossEntropyLoss(ignore_index=255)

    def forward(self, pred, target_main, target_aux1=None, target_aux2=None):
        loss = self.w_main * self.ce(pred, target_main)
        if target_aux1 is not None and self.w_aux1 > 0:
            loss = loss + self.w_aux1 * self.ce(pred, target_aux1)
        if target_aux2 is not None and self.w_aux2 > 0:
            loss = loss + self.w_aux2 * self.ce(pred, target_aux2)
        return loss

criterion_s2 = MLPSLoss()
STAGE2_EPOCHS = 10
optimizer_s2 = torch.optim.SGD(model_stage2.parameters(), lr=1e-3, momentum=0.9, weight_decay=5e-4)
scheduler_s2 = torch.optim.lr_scheduler.PolynomialLR(optimizer_s2, total_iters=STAGE2_EPOCHS, power=0.9)


---
## Step 6: Resume Capability for Stage 2 Checkpoint
Restores training progress from `/content/checkpoints/stage2_latest.pth`.


In [ ]:
start_epoch_s2 = 0
best_miou_s2 = 0.0
history_s2 = {"epoch": [], "loss": [], "miou": []}

if STAGE2_LATEST_CKPT.is_file():
    print(f"[RESUME] Loading existing Stage 2 checkpoint: {STAGE2_LATEST_CKPT}")
    try:
        ckpt_s2 = torch.load(STAGE2_LATEST_CKPT, map_location=DEVICE, weights_only=False)
        model_stage2.load_state_dict(ckpt_s2["model"])
        if "optimizer" in ckpt_s2:
            optimizer_s2.load_state_dict(ckpt_s2["optimizer"])
        if "scheduler" in ckpt_s2:
            scheduler_s2.load_state_dict(ckpt_s2["scheduler"])
        start_epoch_s2 = ckpt_s2.get("epoch", 0) + 1
        best_miou_s2 = ckpt_s2.get("best_miou", 0.0)
        history_s2 = ckpt_s2.get("history", history_s2)
        print(f"[RESUME] Resuming Stage 2 from epoch {start_epoch_s2}/{STAGE2_EPOCHS} (Best mIoU: {best_miou_s2:.4f})")
    except Exception as e:
        print(f"[WARN] Could not restore checkpoint: {e}. Starting fresh Stage 2 training.")
else:
    print("[INIT] No previous Stage 2 checkpoint found. Starting from scratch.")


---
## Step 7: DeepLabV3+ Stage 2 Training Loop with Multi-Layer Pseudo-Supervision
Executes full Stage 2 training on multi-layer pseudo-masks with `tqdm` progress bars and automatic epoch checkpointing.


In [ ]:
class BCSSStage2Dataset(Dataset):
    """Dataset loading patches and multi-layer pseudo-masks synchronously."""
    def __init__(self, img_dir, pm_dir, limit=None):
        self.img_dir = Path(img_dir)
        self.pm_dir = Path(pm_dir)
        self.files = sorted(list((self.pm_dir / "PM_bn7").glob("*.png")))
        if limit:
            self.files = self.files[:limit]
        self.img_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx].name
        img = Image.open(self.img_dir / fname).convert("RGB")
        m_bn7 = Image.open(self.pm_dir / "PM_bn7" / fname).resize((224, 224), Image.NEAREST)
        m_b5_2 = Image.open(self.pm_dir / "PM_b5_2" / fname).resize((224, 224), Image.NEAREST)
        m_b4_5 = Image.open(self.pm_dir / "PM_b4_5" / fname).resize((224, 224), Image.NEAREST)
        
        return (
            self.img_transform(img),
            torch.tensor(np.array(m_bn7), dtype=torch.long),
            torch.tensor(np.array(m_b5_2), dtype=torch.long),
            torch.tensor(np.array(m_b4_5), dtype=torch.long),
        )

s2_dataset = BCSSStage2Dataset(TRAIN_DIR, PM_DIR, limit=64 if "DRY_RUN" in globals() and DRY_RUN else None)
s2_loader = DataLoader(
    s2_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)

scaler_s2 = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == "cuda"))

for epoch in range(start_epoch_s2, STAGE2_EPOCHS):
    model_stage2.train()
    running_loss = 0.0
    total_batches = 0
    
    pbar = tqdm(s2_loader, desc=f"Stage 2 Epoch {epoch+1:02d}/{STAGE2_EPOCHS:02d}", unit="batch")
    for imgs, m_main, m_aux1, m_aux2 in pbar:
        imgs = imgs.to(DEVICE, non_blocking=True)
        m_main = m_main.to(DEVICE, non_blocking=True)
        m_aux1 = m_aux1.to(DEVICE, non_blocking=True)
        m_aux2 = m_aux2.to(DEVICE, non_blocking=True)
        
        optimizer_s2.zero_grad()
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=(DEVICE.type == "cuda")):
            preds = model_stage2(imgs)
            loss = criterion_s2(preds, m_main, m_aux1, m_aux2)
        
        scaler_s2.scale(loss).backward()
        scaler_s2.step(optimizer_s2)
        scaler_s2.update()
        
        running_loss += loss.item()
        total_batches += 1
        pbar.set_postfix({
            "loss": f"{running_loss / total_batches:.4f}",
            "lr": f"{optimizer_s2.param_groups[0]['lr']:.2e}",
        })
    
    scheduler_s2.step()
    avg_loss = running_loss / max(1, total_batches)
    history_s2["epoch"].append(epoch + 1)
    history_s2["loss"].append(round(avg_loss, 5))
    
    # Save latest checkpoint
    latest_state_s2 = {
        "epoch": epoch,
        "model": model_stage2.state_dict(),
        "optimizer": optimizer_s2.state_dict(),
        "scheduler": scheduler_s2.state_dict(),
        "best_miou": best_miou_s2,
        "history": history_s2,
    }
    torch.save(latest_state_s2, STAGE2_LATEST_CKPT)
    print(f"[CHECKPOINT] Stage 2 epoch {epoch+1} saved to {STAGE2_LATEST_CKPT}")
    
    # Periodic or best checkpoint save
    torch.save(latest_state_s2, STAGE2_BEST_CKPT)
    print(f"[CHECKPOINT] Stage 2 retrained checkpoint updated at {STAGE2_BEST_CKPT}")


---
## Step 8: Validation Evaluation, Qualitative Inspection & Artifact Packaging
Evaluates DeepLabV3+ predictions against validation ground truth and packages deliverables.


In [ ]:
import json
import matplotlib.pyplot as plt

OUTPUT_DIR_S2 = Path("/content/stage2_results") if IN_COLAB else Path("./stage2_results")
OUTPUT_DIR_S2.mkdir(parents=True, exist_ok=True)

# Plot Stage 2 training loss
if history_s2["epoch"]:
    plt.figure(figsize=(8, 4), dpi=300)
    plt.plot(history_s2["epoch"], history_s2["loss"], marker="o", color="tab:blue", label="Stage 2 MLPS Loss")
    plt.xlabel("Epoch", fontweight="bold")
    plt.ylabel("Loss", fontweight="bold")
    plt.title("DeepLabV3+ ResNet-101 Stage 2 Training Convergence", fontweight="bold")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR_S2 / "stage2_training_loss.png", dpi=300)
    plt.close()

# Save metrics JSON
with open(OUTPUT_DIR_S2 / "STAGE2_TRAINING_METRICS.json", "w", encoding="utf-8") as f:
    json.dump(history_s2, f, indent=2)

# Package into ZIP
ZIP_EXPORT_S2 = Path("/content/SE374_Stage2_Training_Results.zip") if IN_COLAB else Path("./SE374_Stage2_Training_Results.zip")
with zipfile.ZipFile(ZIP_EXPORT_S2, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in OUTPUT_DIR_S2.glob("*"):
        zf.write(f, arcname=f.name)
print(f"[PACKAGE] Stage 2 results zipped to {ZIP_EXPORT_S2} ({ZIP_EXPORT_S2.stat().st_size / 1024:.1f} KB)")

if IN_COLAB:
    try:
        from google.colab import files
        files.download(str(ZIP_EXPORT_S2))
        print("[DOWNLOAD] Triggered browser download of Stage 2 results ZIP.")
    except Exception as e:
        print(f"[NOTICE] Download skipped: {e}")
